# RARE25 FCDD — Colab Notebook

**Run Sections A–D every session.**  
**Run Section E once ever** (downloads data + backbone to Drive).  
**Run Section F to train / Section G to evaluate.**

Before starting: `Runtime → Change runtime type → T4 GPU`

---
## A · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/rare25-project'
print('Drive root:', DRIVE_ROOT)

---
## B · Clone / pull the GitHub repo

**First session:** uncomment the `git clone` line and fill in your repo URL.  
**Later sessions:** leave only the `git pull` line active.

In [ ]:
import os
os.chdir('/content')

REPO_URL = 'https://github.com/tri2911/rare25-fcdd.git'
REPO_DIR = '/content/rare25-fcdd'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

---
## C · Install dependencies

In [ ]:
!pip install -q timm huggingface_hub scikit-learn matplotlib opencv-python pyyaml
print('Dependencies installed.')

---
## D · Verify GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU — go to Runtime → Change runtime type → T4 GPU')

---
## E · One-time data setup  *(run once, ever)*

Everything saves to Drive permanently. Skip this section on subsequent sessions.

### E1 · Download RARE25 dataset from HuggingFace

In [ ]:
from huggingface_hub import snapshot_download, login
from google.colab import userdata
import os

# Read HF_TOKEN from Colab secrets (Key icon in left sidebar)
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=False)

raw_dir = os.path.join(DRIVE_ROOT, 'data', 'raw')
os.makedirs(raw_dir, exist_ok=True)

snapshot_download(
    repo_id='TimJaspersTue/RARE25-train',
    repo_type='dataset',
    local_dir=raw_dir,
    token=hf_token,
)
print('Download complete. Files saved to:', raw_dir)

### E2 · Download GastroNet / SurgeNet backbone

In [ ]:
from huggingface_hub import hf_hub_download
from google.colab import userdata
import os

hf_token     = userdata.get('HF_TOKEN')
backbone_dir = os.path.join(DRIVE_ROOT, 'backbones')
os.makedirs(backbone_dir, exist_ok=True)

dest = os.path.join(backbone_dir, 'SurgeNet_teacher.pth')

if not os.path.exists(dest):
    print('Downloading GastroNet checkpoint (~100 MB)...')
    hf_hub_download(
        repo_id='TimJaspersTue/SurgeNetModels',
        filename='SurgeNet_checkpoint_epoch0050_teacher.pth',
        repo_type='model',
        local_dir=backbone_dir,
        token=hf_token,
    )
    # hf_hub_download saves with the original filename, rename to match config
    src = os.path.join(backbone_dir, 'SurgeNet_checkpoint_epoch0050_teacher.pth')
    if src != dest:
        os.rename(src, dest)
    print('Saved to:', dest)
else:
    print('Already exists, skipping.')

### E3 · Create train / val / test splits

In [ ]:
import os

raw_dir    = os.path.join(DRIVE_ROOT, 'data', 'raw')
splits_dir = os.path.join(DRIVE_ROOT, 'data', 'splits')

!python src/data_pipeline.py \
    --raw_dir    {raw_dir} \
    --splits_dir {splits_dir} \
    --val_frac  0.15 \
    --test_frac 0.15 \
    --seed      42

### E4 · Verify everything is in place

In [ ]:
import os

paths_to_check = [
    f'{DRIVE_ROOT}/data/raw',
    f'{DRIVE_ROOT}/data/splits/train.csv',
    f'{DRIVE_ROOT}/data/splits/val.csv',
    f'{DRIVE_ROOT}/data/splits/test.csv',
    f'{DRIVE_ROOT}/backbones/SurgeNet_teacher.pth',
]

all_ok = True
for path in paths_to_check:
    status = 'OK     ' if os.path.exists(path) else 'MISSING'
    if status == 'MISSING':
        all_ok = False
    print(f'[{status}]  {path}')

print()
print('Ready to train!' if all_ok else 'Fix missing items above before training.')

---
## F · Training  *(run every session)*

In [ ]:
!python src/train.py \
    --config     config.yaml \
    --drive_root {DRIVE_ROOT}

---
## G · Evaluate on test set

In [ ]:
import glob, os

ckpt_dir   = os.path.join(DRIVE_ROOT, 'outputs', 'checkpoints')
best_ckpt  = os.path.join(ckpt_dir, 'best_model.pth')

# Fall back to latest numbered checkpoint if best_model doesn't exist yet
if not os.path.exists(best_ckpt):
    numbered = sorted(glob.glob(os.path.join(ckpt_dir, 'epoch_*.pth')))
    best_ckpt = numbered[-1] if numbered else None

print('Evaluating checkpoint:', best_ckpt)

if best_ckpt:
    !python src/evaluate.py \
        --config     config.yaml \
        --drive_root {DRIVE_ROOT} \
        --checkpoint {best_ckpt} \
        --split      test

---
## H · Generate anomaly heatmaps

In [ ]:
import glob, os

ckpt_dir  = os.path.join(DRIVE_ROOT, 'outputs', 'checkpoints')
best_ckpt = os.path.join(ckpt_dir, 'best_model.pth')

if not os.path.exists(best_ckpt):
    numbered = sorted(glob.glob(os.path.join(ckpt_dir, 'epoch_*.pth')))
    best_ckpt = numbered[-1] if numbered else None

print('Generating heatmaps from:', best_ckpt)

if best_ckpt:
    !python src/visualise.py \
        --config     config.yaml \
        --drive_root {DRIVE_ROOT} \
        --checkpoint {best_ckpt} \
        --split      test \
        --n          50